# ISTAT Traffic Analysis

-Importazione degli strumenti: Carica le librerie che useremo

-Struttura delle cartelle: Definisce e crea le directory per i file grezzi scaricati e per i dati puliti

In [2]:
# Imports
from pathlib import Path
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

project_folder_raw = Path("data/raw")
project_folder_clean = Path("data/clean")

# Just to be sure the folders are there, preventing FileNotFoundError
project_folder_raw.mkdir(parents=True, exist_ok=True)
project_folder_clean.mkdir(parents=True, exist_ok=True)

# check
# print("Raw folder:", project_folder_raw)
# print("clean folder:", project_folder_clean)

In [3]:
#ISTAT endpoint, csv conv. and final location
ISTAT_URL = "https://esploradati.istat.it/SDMXWS/rest/data/41_983"
HEADERS = {"Accept": "application/vnd.sdmx.data+csv;version=1.0.0"}
ISTAT_path_raw = project_folder_raw / "istat__raw.csv"

try:
    response = requests.get(ISTAT_URL, headers=HEADERS, timeout=20)
    # Erros signaling 
    response.raise_for_status()
    # Save CSV
    ISTAT_path_raw.write_bytes(response.content)
    print("Downloaded ISTAT file:", ISTAT_path_raw)
    print("HTTP status:", response.status_code)
except requests.exceptions.RequestException as e:
    raise RuntimeError("API failed") from e

RuntimeError: API failed

Check and Iinspection of my raw data

In [4]:
# dtype=str typecast
istat_raw = pd.read_csv(ISTAT_path_raw, dtype=str)

# print("Rows:", len(istat_raw))

#inspection
# print("Columns:", list(istat_raw.columns))

# check.
# display(istat_raw.head(15))

istat_raw.info()


<class 'pandas.DataFrame'>
RangeIndex: 573552 entries, 0 to 573551
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   DATAFLOW          573552 non-null  str  
 1   FREQ              573552 non-null  str  
 2   REF_AREA          573552 non-null  str  
 3   DATA_TYPE         573552 non-null  str  
 4   RESULT            573552 non-null  str  
 5   TIME_PERIOD       573552 non-null  str  
 6   OBS_VALUE         573552 non-null  str  
 7   OBS_STATUS        0 non-null       str  
 8   NOTE_DS           0 non-null       str  
 9   NOTE_REF_AREA     0 non-null       str  
 10  NOTE_DATA_TYPE    0 non-null       str  
 11  NOTE_RESULT       0 non-null       str  
 12  NOTE_TIME_PERIOD  0 non-null       str  
 13  BASE_PER          0 non-null       str  
 14  UNIT_MEAS         0 non-null       str  
 15  UNIT_MULT         0 non-null       str  
dtypes: str(16)
memory usage: 70.0 MB


Accidents

In [8]:
# Incidenti, morti e feriti - comuni
# ROADACC copy
istat_acc = istat_raw[istat_raw["DATA_TYPE"] == "ROADACC"].copy()

#conversion to num (all int)
istat_acc["TIME_PERIOD"] = pd.to_numeric(istat_acc["TIME_PERIOD"])
istat_acc["OBS_VALUE"] = pd.to_numeric(istat_acc["OBS_VALUE"])

# conrolla as_index=False per evitare che diventino il nome della riga"
istat_yearly = istat_acc.groupby(["REF_AREA", "TIME_PERIOD"], as_index=False)["OBS_VALUE"].sum()

# renaming columns
istat_yearly = istat_yearly.rename(columns={"REF_AREA": "area_id", "TIME_PERIOD": "year", "OBS_VALUE": "total_accidents"})


istat_yearly.to_csv(project_folder_clean / "istat_accidents_yearly_simple.csv", index=False)

# check
print("Created:", project_folder_clean / "istat_accidents_yearly_simple.csv")
display(istat_yearly.head(11))




# istat_check = pd.read_csv(project_folder_clean / "istat_accidents_yearly_simple.csv", dtype=str)
# istat_check.info()

Created: data\clean\istat_accidents_yearly_simple.csv


,area_id,year,total_accidents
0,001001,2001,5
1,001001,2002,5
2,001001,2003,4
3,001001,2004,9
4,001001,2005,2
5,001001,2006,1
6,001001,2007,8
7,001001,2008,5
8,001001,2009,4
9,001001,2010,7
